# 05 · The Future — Multimodal Models & Agents
### *Ethics, Safety & the Future of LLMs — Unit 3*

Two frontiers: models that **see** (multimodal) and models that **act** (agents). We build a small, real example of each.

1. **CLIP** — zero-shot image understanding (a multimodal model).
2. A tiny **ReAct agent** with tools, plus the **guardrails** that keep it safe.

> CPU is fine.

In [ ]:
!pip -q install "transformers>=4.40" scikit-image pillow torch

## 1 · Multimodal: zero-shot vision with CLIP

CLIP maps **images and text into one shared space**, so you can classify an image against arbitrary text labels it
was never explicitly trained on. This is the seeing-and-reading capability behind modern multimodal LLMs.

In [ ]:
from transformers import CLIPModel, CLIPProcessor
from skimage import data
from PIL import Image
import torch

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
proc  = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

images = {"astronaut": data.astronaut(), "coffee cup": data.coffee(), "cat": data.chelsea()}
labels = ["an astronaut", "a cup of coffee", "a cat", "a dog", "a city street", "a mountain"]

for name, arr in images.items():
    img = Image.fromarray(arr)
    inputs = proc(text=labels, images=img, return_tensors="pt", padding=True)
    with torch.no_grad():
        probs = model(**inputs).logits_per_image.softmax(dim=1)[0]
    best = int(probs.argmax())
    print(f"{name:11s} -> \"{labels[best]}\"  (confidence {probs[best]:.2f})")

CLIP correctly matches each image to a text label with no task-specific training — the core trick that lets an LLM
"look" at an image. **New risk:** the same open-endedness enables *visual jailbreaks* and deepfakes that text-only
filters miss.

## 2 · Agents: a tiny ReAct loop with tools

An agent wraps a controller in a **Think → Act → Observe** loop with tools. We keep the planner simple and
deterministic so the safety mechanics are clear.

In [ ]:
import re

# --- tools ---
def calculator(expr):
    if not re.fullmatch(r"[0-9\.\s()+\-*/]+", expr):     # only arithmetic allowed
        return "refused: invalid expression"
    return str(eval(expr, {"__builtins__": {}}, {}))

FACTS = {"eiffel tower height": "330 metres", "speed of light": "299,792 km/s"}
def lookup(query):
    return FACTS.get(query.lower().strip(), "not found")

TOOLS = {"calculator": calculator, "lookup": lookup}

# --- controller (very small, rule-based planner) ---
def plan(question):
    if re.search(r"[0-9].*[+\-*/].*[0-9]", question):
        expr = re.search(r"[0-9\.\s()+\-*/]+", question).group()
        return "calculator", expr.strip()
    return "lookup", question.replace("What is the ", "").replace("?", "").strip()

def agent(question, allowed_tools, max_steps=3):
    for step in range(max_steps):
        tool, arg = plan(question)                       # THINK
        if tool not in allowed_tools:                    # guardrail: allow-list
            return f"Refused: tool '{tool}' is not allowed."
        obs = TOOLS[tool](arg)                           # ACT + OBSERVE
        print(f"  step {step+1}: think->use {tool}('{arg}')  observe-> {obs}")
        return f"Answer: {obs}"
    return "Stopped: step limit reached."

print(agent("What is 42 * (7 + 3)?", allowed_tools={"calculator","lookup"}))
print(agent("What is the eiffel tower height?", allowed_tools={"calculator","lookup"}))

## 3 · Guardrails in action

Autonomy is dangerous when an agent can take **real, irreversible actions**. We add a `send_email` tool that is
**not on the allow-list** and requires **human approval** — and watch the guardrail refuse it.

In [ ]:
def send_email(to):        # a "dangerous" action tool
    return f"(email sent to {to})"
TOOLS["send_email"] = send_email

def guarded_agent(action, arg, allowed_tools, human_approved=False):
    if action not in allowed_tools:
        return f"⛔ Blocked: '{action}' is not in the allow-list."
    if action == "send_email" and not human_approved:
        return "⏸ Held for human approval before sending."
    return TOOLS[action](arg)

print(guarded_agent("send_email", "ceo@corp.com", allowed_tools={"calculator","lookup"}))
print(guarded_agent("send_email", "ceo@corp.com", allowed_tools={"calculator","lookup","send_email"}))
print(guarded_agent("send_email", "ceo@corp.com", allowed_tools={"calculator","lookup","send_email"}, human_approved=True))

## Recap & your turn

- **Multimodal** models (CLIP-style) share one space for images and text — powerful, with a new safety surface.
- **Agents** = LLM + tools + memory in a loop; they can plan and act.
- **Guardrails** — allow-lists, step limits, and human approval for high-impact actions — are essential, not optional.

**Exercises**
1. Add a `search` tool that returns *untrusted* text, then defend the agent against injection (reuse Notebook 04).
2. Give the planner to `flan-t5` and let the model choose the tool.
3. Log every step to build an **audit trail** — a core requirement for deploying agents safely.